#### Verbindung zu MongoDB herstellen

In [1]:
from pymongo import MongoClient

# Verbindung zum lokalen MongoDB-Server
client = MongoClient('mongodb://localhost:27017/')

# Datenbank auswählen
db = client['meine_datenbank']

# Collection auswählen
collection = db['inventory']

#### Beispieldaten einfügen

In [2]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['shop_db']
inventory = db['inventory']

# Alte Daten löschen (für sauberen Start)
inventory.delete_many({})

# Beispieldaten einfügen
dokumente = [
    {
        "item": "journal",
        "qty": 25,
        "size": {"h": 14, "w": 21, "uom": "cm"},
        "status": "A",
        "tags": ["blank", "red"]
    },
    {
        "item": "notebook",
        "qty": 50,
        "size": {"h": 8.5, "w": 11, "uom": "in"},
        "status": "A",
        "tags": ["red", "blank"]
       },
    {
        "item": "paper",
        "qty": 100,
        "size": {"h": 8.5, "w": 11, "uom": "in"},
        "status": "D",
        "tags": ["red", "blank", "plain"]
    },
    {
        "item": "planner",
        "qty": 75,
        "size": {"h": 22.85, "w": 30, "uom": "cm"},
        "status": "D",
        "tags": ["blank", "red"]
    },
    {
        "item": "postcard",
        "qty": 45,
        "size": {"h": 10, "w": 15.25, "uom": "cm"},
        "status": "A",
        "tags": ["blue"]
    }
]

result = inventory.insert_many(dokumente)
print(f"{len(result.inserted_ids)} Dokumente eingefügt")

5 Dokumente eingefügt


##### Cursor-Methoden

Die find()-Methode gibt keinen direkten Datensatz zurück, sondern einen Cursor. Ein Cursor ist ein Zeiger auf die Ergebnismenge, der es ermöglicht, durch die Dokumente zu iterieren und verschiedene Operationen anzuwenden.

Sortieren mit sort()

In [3]:
# Nach Menge aufsteigend sortieren (1 = aufsteigend)
ergebnis = inventory.find().sort("qty", 1)

print("Nach Menge aufsteigend sortiert:")
for dok in ergebnis:
    print(f"- {dok['item']}: {dok['qty']} Stück")

Nach Menge aufsteigend sortiert:
- journal: 25 Stück
- postcard: 45 Stück
- notebook: 50 Stück
- planner: 75 Stück
- paper: 100 Stück


In [4]:
# Nach Menge absteigend sortieren (-1 = absteigend)
ergebnis = inventory.find().sort("qty", -1)

print("Nach Menge absteigend sortiert:")
for dok in ergebnis:
    print(f"- {dok['item']}: {dok['qty']} Stück")

Nach Menge absteigend sortiert:
- paper: 100 Stück
- planner: 75 Stück
- notebook: 50 Stück
- postcard: 45 Stück
- journal: 25 Stück


Alternative Syntax mit pymongo

In [5]:
from pymongo import ASCENDING, DESCENDING

# Aufsteigend
ergebnis = inventory.find().sort("qty", ASCENDING)

print("Mit ASCENDING:")
for dok in ergebnis:
    print(f"- {dok['item']}: {dok['qty']}")
    
# Absteigend
ergebnis = inventory.find().sort("qty", DESCENDING)

print("Mit DESCENDING:")
for dok in ergebnis:
    print(f"- {dok['item']}: {dok['qty']}")

Mit ASCENDING:
- journal: 25
- postcard: 45
- notebook: 50
- planner: 75
- paper: 100
Mit DESCENDING:
- paper: 100
- planner: 75
- notebook: 50
- postcard: 45
- journal: 25


Nach mehreren Felder sortieren

In [6]:
# Erst nach status, dann nach qty
ergebnis = inventory.find().sort([
    ("status", ASCENDING),
    ("qty", DESCENDING)
])

print("Nach Status (aufsteigend), dann Menge (absteigend):")
for dok in ergebnis:
    print(f"- {dok['item']}: Status={dok['status']}, Menge={dok['qty']}")

Nach Status (aufsteigend), dann Menge (absteigend):
- notebook: Status=A, Menge=50
- postcard: Status=A, Menge=45
- journal: Status=A, Menge=25
- paper: Status=D, Menge=100
- planner: Status=D, Menge=75


Nach verschachtelten Feldern sortieren

In [7]:
# Nach Höhe im size-Objekt sortieren
ergebnis = inventory.find().sort("size.h", DESCENDING)

print("Nach Höhe absteigend:")
for dok in ergebnis:
    print(f"- {dok['item']}: Höhe {dok['size']['h']} {dok['size']['uom']}")

Nach Höhe absteigend:
- planner: Höhe 22.85 cm
- journal: Höhe 14 cm
- postcard: Höhe 10 cm
- notebook: Höhe 8.5 in
- paper: Höhe 8.5 in


##### 2. Limit - Anzahl begrenzen

Die ersten N Dokumente abrufen

In [8]:
# Nur die ersten 3 Dokumente
ergebnis = inventory.find().limit(3)

print("Erste 3 Dokumente:")
for dok in ergebnis:
    print(f"- {dok['item']}")

Erste 3 Dokumente:
- journal
- notebook
- paper


Limit mit Sortierung kombinieren

In [9]:
# Die 3 Artikel mit der höchsten Menge
ergebnis = inventory.find().sort("qty", DESCENDING).limit(3)

print("Top 3 nach Menge:")
for dok in ergebnis:
    print(f"- {dok['item']}: {dok['qty']} Stück")

Top 3 nach Menge:
- paper: 100 Stück
- planner: 75 Stück
- notebook: 50 Stück


Limit mit Filter kombinieren

In [10]:
# Die ersten 2 Artikel mit Status "A"
ergebnis = inventory.find({"status": "A"}).limit(2)

print("Erste 2 mit Status A:")
for dok in ergebnis:
    print(f"- {dok['item']}")

Erste 2 mit Status A:
- journal
- notebook


##### 3. Dokumente überspringen (Skip)

In [11]:
# Überspringe die ersten 2 Dokumente
ergebnis = inventory.find().skip(2)

print("Ab dem 3. Dokument:")
for dok in ergebnis:
    print(f"- {dok['item']}")

Ab dem 3. Dokument:
- paper
- planner
- postcard


Pagination (Seitenweise Anzeige)

In [11]:
def hole_seite(seiten_nummer, pro_seite=2):
    """
    Holt eine bestimmte Seite von Ergebnissen
    
    Args:
        seiten_nummer: Seitennummer (beginnt bei 1)
        pro_seite: Anzahl Dokumente pro Seite
    """
    skip_anzahl = (seiten_nummer - 1) * pro_seite
    
    ergebnis = inventory.find().skip(skip_anzahl).limit(pro_seite)
    
    return list(ergebnis)

# Seite 1 (Dokumente 1-2)
print("Seite 1:")
for dok in hole_seite(1, 2):
    print(f"- {dok['item']}")

print("\nSeite 2:")
for dok in hole_seite(2, 2):
    print(f"- {dok['item']}")

print("\nSeite 3:")
for dok in hole_seite(3, 2):
    print(f"- {dok['item']}")

print("\nSeite 4:")
for dok in hole_seite(4, 2):
    print(f"- {dok['item']}")


Seite 1:
- journal
- notebook

Seite 2:
- paper
- planner

Seite 3:
- postcard

Seite 4:


Skip, Limit und Sort kombinieren

In [13]:
# Überspringe die Top 2, zeige die nächsten 2
ergebnis = inventory.find().sort("qty", DESCENDING).skip(2).limit(2)

print("Platz 3-4 nach Menge:")
for i, dok in enumerate(ergebnis, start=3):
    print(f"{i}. {dok['item']}: {dok['qty']} Stück")

Platz 3-4 nach Menge:
3. notebook: 50 Stück
4. postcard: 45 Stück


##### 4. Dokumente zählen (Count)

In [14]:
# Gesamtanzahl in der Collection
anzahl = inventory.count_documents({})
print(f"Gesamtanzahl Dokumente: {anzahl}")

Gesamtanzahl Dokumente: 5


Anzahl mit Filter

In [15]:
# Anzahl Dokumente mit Status "A"
anzahl_a = inventory.count_documents({"status": "A"})
print(f"Dokumente mit Status A: {anzahl_a}")

# Anzahl Dokumente mit Menge > 50
anzahl_gross = inventory.count_documents({"qty": {"$gt": 50}})
print(f"Dokumente mit Menge > 50: {anzahl_gross}")


Dokumente mit Status A: 3
Dokumente mit Menge > 50: 2


Geschätzte Anzahl (schneller)

In [ ]:
# Schnelle Schätzung (verwendet Metadaten)
geschaetzt = inventory.estimated_document_count()
print(f"Geschätzte Anzahl: {geschaetzt}")

Geschätzte Anzahl: 5


##### 5. Eindeutige Werte (Distinct)

Eindeutige Werte eines Feldes

In [17]:
# Alle eindeutigen Status-Werte
status_werte = inventory.distinct("status")
print(f"Eindeutige Status-Werte: {status_werte}")


Eindeutige Status-Werte: ['A', 'D']


Distinct bei Arrays

In [18]:
# Alle eindeutigen Tags
alle_tags = inventory.distinct("tags")
print(f"Alle Tags: {alle_tags}")

Alle Tags: ['blank', 'blue', 'plain', 'red']


##### 6. Cursor in Liste konvertieren

In [22]:
from pprint import pprint

# Cursor in Liste umwandeln
ergebnis_liste = list(inventory.find({"status": "A"}))

print(f"Anzahl Ergebnisse: {len(ergebnis_liste)}")
print("\nErstes Ergebnis:")
pprint(ergebnis_liste[0])

Anzahl Ergebnisse: 3

Erstes Ergebnis:
{'_id': ObjectId('69c02f0d403fe6c3419f0af3'),
 'item': 'journal',
 'qty': 25,
 'size': {'h': 14, 'uom': 'cm', 'w': 21},
 'status': 'A',
 'tags': ['blank', 'red']}


Nur bestimmte Felder extrahieren

In [25]:
# Liste mit nur den Item-Namen
items = [dok['item'] for dok in inventory.find({"status": "A"})]
print(f"Items mit Status A: {items}")

Items mit Status A: ['journal', 'notebook', 'postcard']


Dictionary erstellen

In [13]:
# Dictionary: item -> qty
mengen_dict = {
    dok['item']: dok['qty'] 
    for dok in inventory.find()
}

print("Mengen-Dictionary:")
for item, qty in mengen_dict.items():
    print(f"  {item}: {qty}")

Mengen-Dictionary:
  journal: 25
  notebook: 50
  paper: 100
  planner: 75
  postcard: 45


##### 7. Batch Size - Performance-Optimierung

Der Server sendet die Dokumente in Gruppen (Batches). Dies kann bei großen Datenmengen die Performance verbessern.

In [27]:
# Dokumente in Batches von 2 abrufen
cursor = inventory.find().batch_size(2)

print("Verarbeitung in Batches:")
for dok in cursor:
    print(f"- {dok['item']}")
    # Hier würde die Verarbeitung stattfinden

Verarbeitung in Batches:
- journal
- notebook
- paper
- planner
- postcard


##### 8. Cursor-Methoden verketten

Was macht das folgende Beispiel? Was wird ausgegeben?

In [14]:
# Komplexe Abfrage mit mehreren Cursor-Methoden
ergebnis = (inventory
    .find({"qty": {"$gte": 25}},
          {"item": 1, "qty": 1, "_id": 0})  # Filter
    .sort("qty", DESCENDING)                # Sortierung
    .skip(1)                                # Ersten überspringen
    .limit(3)                               # Nur 3 Ergebnisse
)

print("Komplexe Abfrage:")
for dok in ergebnis:
    print(f"- {dok['item']}: {dok['qty']}")

Komplexe Abfrage:
- planner: 75
- notebook: 50
- postcard: 45


##### 9 .Cursor-Iteration mit Kontext-Manager

Explizites Schließen des Cursors

In [36]:
# Cursor explizit schließen (wichtig bei großen Datenmengen)
cursor = inventory.find()

try:
    for dok in cursor:
        print(f"- {dok['item']}")
        # Verarbeitung...
finally:
    cursor.close()

- journal
- notebook
- paper
- planner
- postcard


Mit Context Manager (empfohlen)

In [37]:
# Automatisches Schließen mit 'with'
with inventory.find() as cursor:
    for dok in cursor:
        print(f"- {dok['item']}")

- journal
- notebook
- paper
- planner
- postcard


##### 10. Cursor-Informationen abfragen

In [41]:
cursor = inventory.find().limit(2)

# Ist der Cursor noch aktiv?
print(f"Cursor alive: {cursor.alive}")

# Alle Dokumente durchlaufen
for dok in cursor:
    pass

# Nach Iteration ist der Cursor nicht mehr aktiv
print(f"Cursor alive nach Iteration: {cursor.alive}")

Cursor alive: True
Cursor alive nach Iteration: False


##### Beispiel einer vollständigen Suche

In [42]:
def suche_artikel(suchbegriff=None, min_menge=None, max_menge=None, 
                  status=None, sortierung="item", limit=10, seite=1):
    """
    Erweiterte Suchfunktion für Artikel
    
    Args:
        suchbegriff: Suche im Item-Namen (Regex)
        min_menge: Minimale Menge
        max_menge: Maximale Menge
        status: Status-Filter
        sortierung: Feld zum Sortieren
        limit: Max. Anzahl Ergebnisse
        seite: Seitennummer für Pagination
    """
    # Query aufbauen
    query = {}
    
    if suchbegriff:
        query["item"] = {"$regex": suchbegriff, "$options": "i"}  # case-insensitive
    
    if min_menge is not None or max_menge is not None:
        query["qty"] = {}
        if min_menge is not None:
            query["qty"]["$gte"] = min_menge
        if max_menge is not None:
            query["qty"]["$lte"] = max_menge
    
    if status:
        query["status"] = status
    
    # Pagination berechnen
    skip_anzahl = (seite - 1) * limit
    
    # Abfrage ausführen
    ergebnis = (inventory
        .find(query)
        .sort(sortierung, ASCENDING)
        .skip(skip_anzahl)
        .limit(limit)
    )
    
    # Gesamtanzahl für Pagination
    gesamt = inventory.count_documents(query)
    
    return {
        "ergebnisse": list(ergebnis),
        "gesamt": gesamt,
        "seite": seite,
        "seiten_gesamt": (gesamt + limit - 1) // limit
    }

# Beispiel-Verwendung
print("Suche: Items mit 'p', Menge 40-80:")
resultat = suche_artikel(
    suchbegriff="p",
    min_menge=40,
    max_menge=80,
    sortierung="qty"
)

print(f"Gefunden: {resultat['gesamt']} Artikel")
print(f"Seite {resultat['seite']} von {resultat['seiten_gesamt']}\n")

for dok in resultat['ergebnisse']:
    print(f"- {dok['item']}: {dok['qty']} Stück, Status {dok['status']}")        

Suche: Items mit 'p', Menge 40-80:
Gefunden: 2 Artikel
Seite 1 von 1

- postcard: 45 Stück, Status A
- planner: 75 Stück, Status D
